<h1>Direct imaging: the planet around beta Pictoris</h1>

In this example we take a look at data obtained with the [NACO](https://www.eso.org/sci/facilities/paranal/decommissioned/naco.html) instrument on the [Very Large Telescope (VLT)](https://www.eso.org/public/unitedkingdom/teles-instr/paranal-observatory/vlt/). These data were described in a paper by [Bonnefoy et al. (2013)](https://arxiv.org/abs/1302.1160); we will be looking at data taken at $4.78~\mu {\rm m}$ (known as the $M'$ band). We follow the example of the [PynPoint](https://pynpoint.readthedocs.io/en/latest/index.html) package (Amara & Quanz 2012; Stolker et al. 2019), with each step written out in a few lines of Python, and look at the planet around beta Pictoris.

<h2>Angular Differential Imaging</h2>

The brightness contrast between star and planet is the main concern with direct imaging. In the lecture notes, we covered coronographs as a possible solution. In this notebook, we cover a different technique called Angular Differential Imaging ([Marois et al. 2006](https://arxiv.org/abs/astro-ph/0512335)). This is a technique that can be used to subtract the light from the central star to very high accuracy. The central star is a point source, and the response of the telescope optics system to a point source is called the Point Spread Function (PSF).

<h3>Field rotation</h3>

When taking long exposures while tracking an object, the field of view will rotate during the exposure due to Earth's rotation; this is called _field rotation_. Normally, this is corrected for during an observation using a field derotator. However, when looking for faint objects around a bright star, turning the derotator off (or, as for these data, setting it to keep the telescope pupil fixed) can be advantageous. The optics of the telescope then keep the same orientation on the detector, so any feature they introduce does not rotate and can therefore be more easily subtracted.

<h3>Accurate Point Spread function</h3>

We now take **many** short observations during one night, so that field rotation can be ignored for a single exposure. Atmospheric turbulence as well as thermal effects are likely to change the PSF during the night, but we can use the large number of observations to track these changes and construct *very* accurate PSFs. These can then be subtracted from each observation.

<h2>Original image set</h2>

Below is an image set of 263 observations of beta Pictoris. The star is in the middle of each image and completely dominates: no planet in sight! You can move the slider to look at different images, and you will see subtle changes. These are all artificial, possibly caused by atmospheric turbulence. (The axes are in the orientation of the detector: as the field rotates, north points in a different direction in each image.)

In [ ]:
%matplotlib inline
import h5py
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.ndimage import rotate

# Figures shown with widgets live in their own output area (widgets.Output).
# show_in() replaces what an output area shows, sending the new content to the browser once.
# (Using 'with out: display(...)' instead sends it back and forth, and at the top level of a
# cell it can make Voila, the app view, hang.)
def show_in(out, obj):
    with out.hold_sync():
        out.outputs = ()
        out.append_display_data(obj)

# Try and make plots look nice
plt.rcParams["font.family"] = 'serif'
plt.rcParams["mathtext.fontset"] = 'stix'
plt.rcParams["font.serif"] = ['Palatino', 'TeX Gyre Pagella', 'DejaVu Serif']   # first available is used
plt.rcParams['pdf.fonttype'] = 42

# The data: 263 images of beta Pictoris (NACO, M' band) and, for each image, the orientation of the
# field on the detector: the parallactic angle plus a fixed offset of the instrument (as prepared for
# the PynPoint tutorial)
with h5py.File('data/betapic_naco_mp.hdf5', 'r') as f:
    stack = f['stack'][...]                           # images: (image, y, x)
    parang = f['header_stack/PARANG'][...]            # field orientation angles (degrees)
    pixscale = float(f['stack'].attrs['PIXSCALE'])    # arcsec per pixel
nimages, ny, nx = stack.shape
size = pixscale*nx/2                                  # half the width of an image (arcsec)


def show_image(out, image, title, sky=False):
    # Show one image; sky=True: north up, east left (after rotating), otherwise detector orientation
    fig, ax = plt.subplots(figsize=(6, 4.5))
    plt.close(fig)
    extent = [size, -size, -size, size] if sky else [-size, size, -size, size]    # R.A. increases to the left
    im = ax.imshow(image, origin='lower', extent=extent)
    ax.set_title(title)
    ax.set_xlabel('R.A. offset [arcsec]' if sky else 'x offset [arcsec]', fontsize=12)
    ax.set_ylabel('Dec. offset [arcsec]' if sky else 'y offset [arcsec]', fontsize=12)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    show_in(out, fig)


style = {'description_width': 'initial'}
w_raw = widgets.IntSlider(value=0, min=0, max=nimages - 1, description='Image index:', continuous_update=False,
                          style=style)
out_raw = widgets.Output()
display(w_raw, out_raw)


def draw_raw(change=None):
    show_image(out_raw, stack[w_raw.value], f"beta Pic - NACO M' - original data (image {w_raw.value})")


w_raw.observe(draw_raw, names='value')
draw_raw()

<h2>Subtracting the saturated centre</h2>

The exposure time was chosen so that the central pixels are completely saturated. If we mask these out (here: everything within 0.15 arcsec of the star, and the corners of the images), the changes from frame to frame become much more apparent. You can now clearly see the ringed structure of the PSF (normal for circular apertures). Still no planet to be seen...

In [ ]:
# Keep pixels between 0.15 arcsec from the star and the edge of the image; set the others to zero
y, x = np.indices((ny, nx))
r = np.hypot(x - (nx - 1)/2, y - (ny - 1)/2)*pixscale          # distance from the star (arcsec)
keep = (r >= 0.15) & (r <= min(1.1, size))
masked = np.where(keep, stack, 0.0)

w_masked = widgets.IntSlider(value=0, min=0, max=nimages - 1, description='Image index:', continuous_update=False,
                             style=style)
out_masked = widgets.Output()
display(w_masked, out_masked)


def draw_masked(change=None):
    show_image(out_masked, masked[w_masked.value], f"beta Pic - NACO M' - centre masked (image {w_masked.value})")


w_masked.observe(draw_masked, names='value')
draw_masked()

<h2>Calculating and subtracting the PSF</h2>

The PSF is modelled using Principal Component Analysis (PCA). It takes the global shapes of the PSF and its variation with time to obtain a very accurate representation of the PSF that can then be subtracted. In practice, the average image is subtracted from every image, and the remaining frame-to-frame changes are described by a set of patterns (the *principal components*), ordered from the most to the least important. Each image is then modelled by its first $k$ components, and this model is subtracted (Amara & Quanz 2012; Soummer, Pueyo & Larkin 2012).

You can see in the individual images that the algorithm did a good job: there is nothing left but noise! The exposure times for the individual frames are too short to reveal the planet.

Use the slider to change the number of components $k$ (20 in the PynPoint example). With $k = 0$ only the average image is subtracted, similar to the original ADI method, which subtracts the median image. With too few components, starlight is left over; with too many, the components start to describe the planet as well, and part of its light is subtracted too: compare the final images below for different $k$.

In [ ]:
# Principal components of the masked images (only the unmasked pixels), after subtracting the average image
data = masked[:, keep]                                  # (image, pixel)
data = data - data.mean(axis=0)
_, _, components = np.linalg.svd(data, full_matrices=False)    # rows: principal components, most important first

w_k = widgets.IntSlider(value=20, min=0, max=100, description='Number of components k:', continuous_update=False,
                        style=style)
w_rotate = widgets.Checkbox(value=True, description='Rotate images before combining', style=style)
w_res = widgets.IntSlider(value=0, min=0, max=nimages - 1, description='Image index:', continuous_update=False,
                          style=style)
out_res = widgets.Output()
display(w_k, w_rotate, w_res, out_res)
S = {}                                                  # results of the PSF subtraction


def subtract_psf():
    # Model each image with the first k components and subtract the model
    pcs = components[:w_k.value]
    residuals = np.zeros_like(masked)
    residuals[:, keep] = data - (data @ pcs.T) @ pcs
    S['residuals'] = residuals
    # Rotate each image back by its field orientation angle: north up and east left in all images
    S['rotated'] = np.array([rotate(im, -angle, reshape=False) for im, angle in zip(residuals, parang)])


def draw_residuals(change=None):
    if w_rotate.value:
        show_image(out_res, S['rotated'][w_res.value], f"beta Pic - NACO M' - residuals, rotated (image {w_res.value})", sky=True)
    else:
        show_image(out_res, S['residuals'][w_res.value], f"beta Pic - NACO M' - residuals (image {w_res.value})")


def update(change=None):
    if change is None or change['owner'] is w_k:
        subtract_psf()
    draw_residuals()
    if 'draw_final' in globals():                       # the combined image, once the next cell has been run
        draw_final()


w_k.observe(update, names='value')
w_rotate.observe(update, names='value')
w_res.observe(draw_residuals, names='value')
update()

<h2>Combining all frames</h2>

If we now combine all frames, we can finally see the planet! The projected separation is roughly 0.5 arcseconds. As it turns out, this technique is much more powerful than taking a single, very long exposure (this would mean only a single, average PSF can be obtained, which is much less accurate). Another drawback is that a longer exposure time would mean much more of the image would get saturated, possibly flooding the planet's location as well (the brightness contrast strikes again!).

Before combining, each image is rotated back by the orientation of the field at the time it was taken (the parallactic angle, plus a fixed offset of about 91 degrees for the orientation of the instrument, which is already included in the angles in the data file). This aligns all images so that North points up and East points to the left in every image. The images are then combined by taking the median of each pixel. During these observations the field rotated by about 51 degrees so, in the original images, the planet appears to move around the star from one image to the next (due to the field's rotation caused by Earth's rotation). Untick *Rotate images before combining* (above the residual images) to see what happens without this step: the planet's light is spread along an arc and the median removes most of it.

In [ ]:
out_final = widgets.Output()
display(out_final)


def draw_final():
    # Median of all residual images (updated by the controls above)
    if w_rotate.value:
        show_image(out_final, np.median(S['rotated'], axis=0),
                   f"beta Pic b - NACO M' - median of the rotated residuals (k = {w_k.value})", sky=True)
    else:
        show_image(out_final, np.median(S['residuals'], axis=0),
                   f"beta Pic - NACO M' - median of the residuals, not rotated (k = {w_k.value})")


draw_final()

<h3>References</h3>

Amara & Quanz 2012, MNRAS 427, 948 (PynPoint) $\cdot$ Bonnefoy et al. 2013, A&A 555, A107 (the beta-Pic data) $\cdot$ Marois et al. 2006, ApJ 641, 556 (angular differential imaging) $\cdot$ Soummer, Pueyo & Larkin 2012, ApJ 755, L28 (PSF subtraction with principal components) $\cdot$ Stolker et al. 2019, A&A 621, A59 (PynPoint).